In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
data_path = 'event_level_data_dirty.csv'
clean_path = 'event_level_data_clean.csv'
df = pd.read_csv(data_path)
df.head()


,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name,timezone_info
0,1,2025-01-01 07:01:00,2.0,7.0,False,True,1.280884,{}103.869885{},rainy,24.8,93.5,0,SEMBAWANG EATING HOUSE,NaN
1,2,2025-01-01 07:16:00,2.0,7.0,\r\nFalse\r\n,\nTrue\n,1.280884,103.869885,rainy,60.0,91.6,0,SEMBAWANG EATING HOUSE,NaN
2,3,NaN,2.0,7.0,False,NaN,1.280884,103.869885,cloudy,()23.7,86.9,0,SEMBAWANG EATING HOUSE,NaN
3,4,2025-01-01 07:38:00,2.0,7.0,@False,True,1.280884,103.869885,cloudy,24.2,85.7,0,SEMBAWANG EATING HOUSE,NaN
4,5,2025-01-01 07:39:00,NaN,7.0,\r\nFalse\r\n,True,\r\n1.280884\r\n,103.869885,rainy,24.7,91().2,0,SEMBAWANG EATING HOUSE,NaN


In [3]:
def clean_data(input_path, output_path):
    # Load data
    df = pd.read_csv(input_path)

    # Drop duplicates and unnecessary columns
    df = df.drop_duplicates(subset=['record_id'], keep='first')
    if 'timezone_info' in df.columns:
        df = df.drop(columns=['timezone_info'])

    # Clean String Noise
    def basic_clean(text):
        if pd.isna(text): return text
        text = str(text).strip().lower()
        text = re.sub(r'[{}()\[\]@*&|°\\ø$%//#/]', '', text)
        text = text.replace('á', 'a').replace('ë', 'e').replace('ï', 'i').replace('ü', 'u').replace('û', 'ou')
        return text

    for col in ['weather', 'is_weekend', 'is_public_holiday', 'location_name']:
        df[col] = df[col].apply(basic_clean)

    def clean_weather(value):
        if pd.isna(value):
            return 'unknown'
        elif 'night_clear' in value:
            return 'night_clear'
        elif 'rain' in value:
            return 'rainy'
        elif 'cloud' in value:
            return 'cloudy'
        elif 'clear' in value:
            return 'clear'
        else:
            return 'other'
    df['weather'] = df['weather'].apply(clean_weather)

    # Handle Timestamps and Date Features
    # Fill missing timestamps based on sequential record_ids
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['timestamp'] = df['timestamp'].interpolate(method='linear')

    # Recalculate time features to ensure consistency
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['hour_of_day'] = df['timestamp'].dt.hour
    df['is_weekend'] = df['day_of_week'].isin([5, 6])

    # Clean Numeric Columns (Lat, Long, Temp, Humidity)
    def clean_numeric(val):
        if pd.isna(val): return np.nan
        # Extract digits, dots, and minus signs only
        clean_val = re.sub(r'[^0-9.\-]', '', str(val))
        try:
            return float(clean_val)
        except:
            return np.nan

    num_cols = ['lat', 'long', 'temperature', 'humidity']
    for col in num_cols:
        df[col] = df[col].apply(clean_numeric)

    # Impute Location Data using Location ID
    # Use location_id to fix incorrect Lat/Long/Name (e.g., -100.0 or 999.0)
    for col in ['lat', 'long', 'location_name']:
        # Create a mapping of location_id to the most common (mode) valid value
        mapping = df[df[col].notna() & (df[col] != 0) & (df[col] != -100) & (df[col] != 999)] \
                    .groupby('location_id')[col].agg(lambda x: x.value_counts().index[0])
        df[col] = df['location_id'].map(mapping)

    # Handle Outliers in Environment Data
    # Cap temperatures to reasonable ranges (e.g., 20-40) or interpolate
    df.loc[(df['temperature'] < 15) | (df['temperature'] > 45), 'temperature'] = np.nan
    df['temperature'] = df.groupby('location_id')['temperature'].transform(lambda x: x.interpolate().ffill().bfill())
    df['humidity'] = df.groupby('location_id')['humidity'].transform(lambda x: x.interpolate().ffill().bfill())

    # Final Formatting
    df['is_public_holiday'] = df['is_public_holiday'].map({'true': True, 'false': False}).fillna(False).astype(bool)
    df['location_id'] = df['location_id'].astype(int)

    # Sort and Save
    df = df.sort_values('record_id')
    df.to_csv(output_path, index=False)
    return df

In [4]:
# Run the cleaning
clean_df = clean_data(data_path, clean_path)

/var/folders/27/7dr1zd1n6bj1vvsl3vy6pg1h0000gn/T/ipykernel_9958/1793197447.py:75: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_public_holiday'] = df['is_public_holiday'].map({'true': True, 'false': False}).fillna(False).astype(bool)


In [5]:
# Extra cleaning patch from clean_data.py's ideas

def fix_bool(val):
    v = str(val).strip().lower()
    return 1 if ("true" in v or v == "1") else 0

# safer timestamp handling
clean_df["timestamp"] = pd.to_datetime(clean_df["timestamp"], errors="coerce").ffill().bfill()

# rebuild time fields from timestamp
clean_df["hour_of_day"] = clean_df["timestamp"].dt.hour
clean_df["day_of_week"] = clean_df["timestamp"].dt.dayofweek

# keep active hours only
clean_df = clean_df[(clean_df["hour_of_day"] >= 6) & (clean_df["hour_of_day"] <= 22)].copy()

# safer numeric cleaning
for col in ["temperature", "humidity"]:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(
            clean_df[col].astype(str).str.replace(r"[^0-9.\-]", "", regex=True),
            errors="coerce"
        )

# Fahrenheit to Celsius
if "temperature" in clean_df.columns:
    clean_df.loc[clean_df["temperature"] > 45, "temperature"] = (
        (clean_df["temperature"] - 32) * 5 / 9
    )

# fill missing values by location median
for col in ["temperature", "humidity"]:
    if col in clean_df.columns:
        clean_df[col] = clean_df.groupby("location_id")[col].transform(
            lambda x: x.fillna(x.median())
        )

# fallback fill
if "temperature" in clean_df.columns:
    clean_df["temperature"] = clean_df["temperature"].fillna(clean_df["temperature"].median())

if "humidity" in clean_df.columns:
    clean_df["humidity"] = clean_df["humidity"].fillna(clean_df["humidity"].median())

# safer boolean conversion
for col in ["is_weekend", "is_public_holiday"]:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].apply(fix_bool).astype(int)

clean_df.head()

,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name
0,1,2025-01-01 07:01:00,2,7,0,1,1.280884,103.869885,rainy,24.80,93.5,0,sembawang eating house
1,2,2025-01-01 07:16:00,2,7,0,1,1.280884,103.869885,rainy,24.25,91.6,0,sembawang eating house
2,3,2025-01-01 07:27:00,2,7,0,0,1.280884,103.869885,cloudy,23.70,86.9,0,sembawang eating house
3,4,2025-01-01 07:38:00,2,7,0,1,1.280884,103.869885,cloudy,24.20,85.7,0,sembawang eating house
4,5,2025-01-01 07:39:00,2,7,0,1,1.280884,103.869885,rainy,24.70,91.2,0,sembawang eating house


In [6]:
clean_df['weather'].unique()

array(['rainy', 'cloudy', 'unknown', 'other', 'clear', 'night_clear'],
      dtype=object)

In [7]:
# Drop unkown(nan) values from weather
clean_df = clean_df[clean_df['weather'] != 'unknown']
clean_df = clean_df[clean_df['weather'] != 'other']
clean_df['weather'].unique()

array(['rainy', 'cloudy', 'clear', 'night_clear'], dtype=object)

In [8]:
# Use dummy variables for weather
weather_map = {'rainy' : 0, 'cloudy': 1, 'night_clear': 2, 'clear': 3}
clean_df['weather_final'] = clean_df['weather'].map(weather_map)
clean_df.head()

,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name,weather_final
0,1,2025-01-01 07:01:00,2,7,0,1,1.280884,103.869885,rainy,24.80,93.5,0,sembawang eating house,0
1,2,2025-01-01 07:16:00,2,7,0,1,1.280884,103.869885,rainy,24.25,91.6,0,sembawang eating house,0
2,3,2025-01-01 07:27:00,2,7,0,0,1.280884,103.869885,cloudy,23.70,86.9,0,sembawang eating house,1
3,4,2025-01-01 07:38:00,2,7,0,1,1.280884,103.869885,cloudy,24.20,85.7,0,sembawang eating house,1
4,5,2025-01-01 07:39:00,2,7,0,1,1.280884,103.869885,rainy,24.70,91.2,0,sembawang eating house,0


In [9]:
# Creating location frequency column for encoding
location_freq = clean_df['location_name'].value_counts()
clean_df['location_freq'] = clean_df['location_name'].map(location_freq)
clean_df.head()

,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name,weather_final,location_freq
0,1,2025-01-01 07:01:00,2,7,0,1,1.280884,103.869885,rainy,24.80,93.5,0,sembawang eating house,0,10073
1,2,2025-01-01 07:16:00,2,7,0,1,1.280884,103.869885,rainy,24.25,91.6,0,sembawang eating house,0,10073
2,3,2025-01-01 07:27:00,2,7,0,0,1.280884,103.869885,cloudy,23.70,86.9,0,sembawang eating house,1,10073
3,4,2025-01-01 07:38:00,2,7,0,1,1.280884,103.869885,cloudy,24.20,85.7,0,sembawang eating house,1,10073
4,5,2025-01-01 07:39:00,2,7,0,1,1.280884,103.869885,rainy,24.70,91.2,0,sembawang eating house,0,10073


In [10]:
import os

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

import mlflow
import mlflow.sklearn

In [11]:
df = clean_df.copy()

print("clean_df rows:", len(df))
print("clean_df columns:", df.columns.tolist())

clean_df rows: 198726
clean_df columns: ['record_id', 'timestamp', 'day_of_week', 'hour_of_day', 'is_weekend', 'is_public_holiday', 'lat', 'long', 'weather', 'temperature', 'humidity', 'location_id', 'location_name', 'weather_final', 'location_freq']


In [12]:
# Create 30 mins time bins
df["time_bin"] = df["timestamp"].dt.floor("30min")

# Quick proof check: should only be 0,30,60,...
print("Minutes in time_bin:", sorted(df["time_bin"].dt.minute.unique())[:12], "...")


Minutes in time_bin: [np.int32(0), np.int32(30)] ...


In [13]:
# Build TARGET y = event_count per (location_id, time_bin)
#    event_count = number of rows in that bin

group_cols = ["location_id", "time_bin"]

df_counts = (
    df.groupby(group_cols)
      .size()
      .reset_index(name="event_count")
)


In [14]:
# Build FEATURES per (location_id, time_bin)

agg_rules = {}
for col, rule in [
    ("day_of_week", "first"),
    ("hour_of_day", "first"),
    ("is_weekend", "first"),
    ("is_public_holiday", "first"),
    ("temperature", "mean"),
    ("humidity", "mean"),
    ("weather", "first"),
    ("location_freq", "first"),
]:
    if col in df.columns:
        agg_rules[col] = rule

df_feat = (
    df.groupby(group_cols, as_index=False)
      .agg(agg_rules)
)

# Merge features + target
df30 = df_feat.merge(df_counts, on=group_cols, how="inner")

# keep location_id as categorical feature
df30["location_id"] = df30["location_id"].astype(str)

# sort by time first for better time-based split
df30 = df30.sort_values(["time_bin", "location_id"]).reset_index(drop=True)

# Create simple time blocks
df30["time_block"] = pd.cut(
    df30["hour_of_day"],
    bins=[6, 12, 17, 19, 23],
    labels=["Morning", "Afternoon", "Evening", "Night"],
    right=False,
    include_lowest=True
)

# Create day names
df30["day_name"] = df30["day_of_week"].map({
    0: "Mon",
    1: "Tue",
    2: "Wed",
    3: "Thu",
    4: "Fri",
    5: "Sat",
    6: "Sun"
})

print("Rows (30-min bins):", len(df30))
print(df30.head())

Rows (30-min bins): 87789
  location_id            time_bin  day_of_week  hour_of_day  is_weekend  \
0           6 2025-01-01 06:00:00            2            6           0   
1           3 2025-01-01 06:30:00            2            6           0   
2           9 2025-01-01 06:30:00            2            6           0   
3           0 2025-01-01 07:00:00            2            7           0   
4           1 2025-01-01 07:00:00            2            7           0   

   is_public_holiday  temperature   humidity weather  location_freq  \
0                  1    24.200000  92.100000   rainy           9931   
1                  0    24.300000  89.500000  cloudy           9841   
2                  1    25.000000  90.550000  cloudy           9960   
3                  1    24.250000  90.666667   rainy          10073   
4                  1    24.566667  91.666667   rainy           9898   

   event_count time_block day_name  
0            1    Morning      Wed  
1            1    Morn

In [15]:
# Create continuous crowd score from event_count

min_count = df30["event_count"].min()
max_count = df30["event_count"].max()

df30["crowd_score"] = (
    (df30["event_count"] - min_count) / (max_count - min_count)
)

y = df30["crowd_score"]

print(df30[["event_count", "crowd_score"]].head())
print(df30["crowd_score"].describe())



   event_count  crowd_score
0            1          0.0
1            1          0.0
2            2          0.1
3            3          0.2
4            3          0.2
count    87789.000000
mean         0.126368
std          0.135083
min          0.000000
25%          0.000000
50%          0.100000
75%          0.200000
max          1.000000
Name: crowd_score, dtype: float64


In [17]:
# Sort back by time before selecting X and y
df30 = df30.sort_values(["time_bin", "location_id"]).reset_index(drop=True)

num_cols = [c for c in [
    "temperature",
    "humidity",
    "is_weekend",
    "is_public_holiday",
    "location_freq"
] if c in df30.columns]

cat_cols = [c for c in ["weather", "location_id", "time_block", "day_name"]
            if c in df30.columns]

X = df30[num_cols + cat_cols].copy()
y = df30["crowd_score"].copy()

for bcol in ["is_weekend", "is_public_holiday"]:
    if bcol in X.columns:
        X[bcol] = X[bcol].astype(int)

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)
print(X.head())
print(y.head())

Numeric columns: ['temperature', 'humidity', 'is_weekend', 'is_public_holiday', 'location_freq']
Categorical columns: ['weather', 'location_id', 'time_block', 'day_name']
   temperature   humidity  is_weekend  is_public_holiday  location_freq  \
0    24.200000  92.100000           0                  1           9931   
1    24.300000  89.500000           0                  0           9841   
2    25.000000  90.550000           0                  1           9960   
3    24.250000  90.666667           0                  1          10073   
4    24.566667  91.666667           0                  1           9898   

  weather location_id time_block day_name  
0   rainy           6    Morning      Wed  
1  cloudy           3    Morning      Wed  
2  cloudy           9    Morning      Wed  
3   rainy           0    Morning      Wed  
4   rainy           1    Morning      Wed  
0    0.0
1    0.0
2    0.1
3    0.2
4    0.2
Name: crowd_score, dtype: float64


In [19]:
split = int(len(df30) * 0.8)

X_train, X_test = X.iloc[:split].copy(), X.iloc[split:].copy()
y_train, y_test = y.iloc[:split].copy(), y.iloc[split:].copy()

print("Train rows:", len(X_train), "Test rows:", len(X_test))
print("\nTrain crowd score summary:")
print(y_train.describe())

print("\nTest crowd score summary:")
print(y_test.describe())

USE_SAMPLE_WEIGHT = False
sample_weight_train = None

Train rows: 70231 Test rows: 17558

Train crowd score summary:
count    70231.000000
mean         0.126587
std          0.134787
min          0.000000
25%          0.000000
50%          0.100000
75%          0.200000
max          1.000000
Name: crowd_score, dtype: float64

Test crowd score summary:
count    17558.000000
mean         0.125493
std          0.136263
min          0.000000
25%          0.000000
50%          0.100000
75%          0.200000
max          0.900000
Name: crowd_score, dtype: float64


In [20]:
# Preprocessing (impute + one-hot)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ],
    remainder="drop"
)


In [21]:
# MLflow setup (stores runs in ./mlruns)
# -----------------------------------------------------------------------
mlflow.set_tracking_uri("file://" + os.path.abspath("mlruns"))
mlflow.set_experiment("crowdedness_30min_eventcount_regression_v2")

/Users/user/Documents/DSA industry project- AIRES/.venv/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
Traceback (most recent call last):
  File "/Users/user/Documents/DSA industry project- AIRES/.venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 379, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "/Users/user/Documents/DSA industry project- AIRES/.venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 477, in _get_experiment
    meta = 

<Experiment: artifact_location=('file:///Users/user/Documents/DSA industry project- '
 'AIRES/mlruns/821697836990436153'), creation_time=1773831209486, experiment_id='821697836990436153', last_update_time=1773831209486, lifecycle_stage='active', name='crowdedness_30min_eventcount_regression_v2', tags={}>

In [22]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

def evaluate_model(run_name, model):
    with mlflow.start_run(run_name=run_name):

        pipe = Pipeline([
            ("prep", preprocess),
            ("model", model)
        ])

        pipe.fit(X_train, y_train, model__sample_weight=sample_weight_train)

        pred = pipe.predict(X_test)
        pred = np.clip(pred, 0, 1)

        mae = mean_absolute_error(y_test, pred)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        r2 = r2_score(y_test, pred)

        mlflow.log_param("model_name", type(model).__name__)
        mlflow.log_metric("mae", float(mae))
        mlflow.log_metric("rmse", float(rmse))
        mlflow.log_metric("r2", float(r2))

        mlflow.sklearn.log_model(pipe, name="model")

        results.append({
            "Model": run_name,
            "MAE": round(mae, 4),
            "RMSE": round(rmse, 4),
            "R2": round(r2, 4)
        })

        print(f"\n{run_name}")
        print(f"MAE  : {mae:.4f}")
        print(f"RMSE : {rmse:.4f}")
        print(f"R2   : {r2:.4f}")


evaluate_model(
    "GradientBoostingRegressor_weighted",
    GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
)

evaluate_model(
    "RandomForestRegressor_weighted",
    RandomForestRegressor(
        n_estimators=800,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )
)

try:
    from xgboost import XGBRegressor

    evaluate_model(
        "XGBRegressor_weighted",
        XGBRegressor(
            n_estimators=1000,
            max_depth=6,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1
        )
    )
except ImportError:
    print("xgboost not installed, skipping XGBRegressor.")

results_df = pd.DataFrame(results)

print("\nFinal regression results:")
print(results_df.to_string(index=False))

/Users/user/Documents/DSA industry project- AIRES/.venv/lib/python3.13/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)



GradientBoostingRegressor_weighted
MAE  : 0.0904
RMSE : 0.1193
R2   : 0.2334


/Users/user/Documents/DSA industry project- AIRES/.venv/lib/python3.13/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)



RandomForestRegressor_weighted
MAE  : 0.0970
RMSE : 0.1246
R2   : 0.1638


/Users/user/Documents/DSA industry project- AIRES/.venv/lib/python3.13/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)



XGBRegressor_weighted
MAE  : 0.0867
RMSE : 0.1181
R2   : 0.2482

Final regression results:
                             Model    MAE   RMSE     R2
GradientBoostingRegressor_weighted 0.0904 0.1193 0.2334
    RandomForestRegressor_weighted 0.0970 0.1246 0.1638
             XGBRegressor_weighted 0.0867 0.1181 0.2482
